# 05 — Batch pipeline: 4 books in parallel, low-storage mode

Processes the whole corpus (or a slice of it) with the production pipeline (`configs/pipeline.toml`):

* **4 books in parallel** — separate *processes* (OCRmyPDF's API is not thread-safe); each book's OCR gets `cpu_count // 4` cores
* **Low-storage mode** — after each book, its downloaded TIFFs, preprocessed pages and the raw pre-OCR PDF are deleted; only the final **PDF + text sidecar** stay in `output/<faculty>/`
* **Disk guard** — a worker waits before downloading if free space drops below `MIN_FREE_GB`
* **Resumable** — books with an existing final PDF are skipped; every finished book is appended to `output/batch_report.jsonl`, so interrupting the kernel loses nothing

Peak disk usage ≈ 4 × (largest in-flight book ≈ TIFFs + pages + raw PDF ≈ 2.5× book size). With the median 6.5 GB book expect ~60–70 GB peaks — tune `MIN_FREE_GB`/`PARALLEL_BOOKS` to your disk.

In [1]:
%load_ext autoreload
%autoreload 2

import json
import os
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

from evilflowers_books_digitalizer import LocalCache, load_settings
from evilflowers_books_digitalizer.batch import expected_pdf, free_gb, process_book

PARALLEL_BOOKS = 4
JOBS_PER_BOOK = max(1, (os.cpu_count() or 4) // PARALLEL_BOOKS)
MIN_FREE_GB = 40.0   # workers pause new downloads below this
FACULTIES = None     # e.g. ["fad", "fei"] — None = all
LIMIT = 10           # books per run — None = everything pending (~days!)

settings = load_settings()
cache = LocalCache(settings.cache_dir)
report_path = settings.output_dir / "batch_report.jsonl"

print(f"{PARALLEL_BOOKS} workers x {JOBS_PER_BOOK} OCR jobs, "
      f"free disk: {free_gb(settings.output_dir.parent):.0f} GB")

4 workers x 2 OCR jobs, free disk: 117 GB


## Work list

All non-empty books from the cached inventory (notebook 01), minus those already digitized. Smallest books first — early failures surface cheaply and disk pressure stays low at the start.

In [2]:
pending, done = [], 0
for key in FACULTIES or settings.sources:
    stats = cache.load_stats(key)
    if stats is None:
        raise RuntimeError("no cached inventory — run notebook 01 first")
    for book in stats.books:
        if book.n_pages == 0:
            continue
        if expected_pdf(settings.output_dir, key, book.book_id).exists():
            done += 1
        else:
            pending.append(book)

pending.sort(key=lambda book: book.total_bytes)
todo = pending[:LIMIT] if LIMIT else pending

print(f"done: {done} | pending: {len(pending)} "
      f"({sum(b.total_bytes for b in pending) / 1e9:,.0f} GB to download)")
print(f"this run: {len(todo)} books, {sum(b.total_bytes for b in todo) / 1e9:.1f} GB, "
      f"{sum(b.n_pages for b in todo)} frames")

done: 9 | pending: 871 (6,355 GB to download)
this run: 10 books, 13.7 GB, 661 frames


## Run

Interrupting the kernel is safe: finished books are already on disk and in the report; in-flight books are re-done on the next run (downloads resume, their cache is cleaned either way).

In [3]:
rows = []
counters = {"ok": 0, "skipped": 0, "error": 0, "gb": 0.0}

with ProcessPoolExecutor(max_workers=PARALLEL_BOOKS) as pool, report_path.open("a") as report:
    futures = {
        pool.submit(
            process_book,
            book.source,
            book.book_id,
            jobs=JOBS_PER_BOOK,
            min_free_gb=MIN_FREE_GB,
        ): book
        for book in todo
    }
    with tqdm(total=len(futures), unit="book", desc="digitizing", smoothing=0.1) as bar:
        for future in as_completed(futures):
            book = futures[future]
            try:
                row = future.result()
            except Exception as exc:  # noqa: BLE001 — even a crashed worker is just a row
                row = {"source": book.source, "book_id": book.book_id,
                       "status": "error", "error": f"worker crash: {exc}"[:500]}
            rows.append(row)
            report.write(json.dumps(row, ensure_ascii=False) + "\n")
            report.flush()

            counters[row["status"]] += 1
            counters["gb"] += row.get("pdf_mb", 0) / 1e3
            bar.update(1)
            bar.set_postfix(ok=counters["ok"], skip=counters["skipped"],
                            err=counters["error"], out=f"{counters['gb']:.1f}GB",
                            disk=f"{free_gb(settings.output_dir.parent):.0f}GB")

            status = row["status"]
            extra = f"{row.get('pdf_mb', 0):.0f} MB, {row.get('language')}, {row.get('minutes', 0):.0f} min" \
                if status == "ok" else row.get("error", "")
            bar.write(f"[{status:>7}] {row['source']}/{row['book_id'][:50]} — {extra}")

digitizing:   0%|          | 0/10 [00:00<?, ?book/s]

INFO evilflowers_books_digitalizer.pipeline.base: pipeline start: fei_CVI_OPACID_FEI_9788022726634 (6 steps)
INFO evilflowers_books_digitalizer.pipeline.base: pipeline start: fei_CVI_OPACID_FEI_9788022726665 (6 steps)
INFO evilflowers_books_digitalizer.pipeline.base: pipeline start: fei_CVI_OPACID_FEI_9788022726658 (6 steps)
INFO evilflowers_books_digitalizer.pipeline.base: pipeline start: fei_CVI_OPACID_FEI_9788022726689 (6 steps)
CVI_OPACID_FEI_9788022726665: 100%|██████████| 53/53 [05:06<00:00,  5.79s/page]
INFO evilflowers_books_digitalizer.pipeline.base: step download         done in  307.8s (fei_CVI_OPACID_FEI_9788022726665)
CVI_OPACID_FEI_9788022726634:  97%|█████████▋| 56/58 [05:24<00:10,  5.16s/page]INFO evilflowers_books_digitalizer.pipeline.steps.preprocess: fei_CVI_OPACID_FEI_9788022726665: 53 frames -> 53 pages (0 spreads split, 53 color pages kept)
INFO evilflowers_books_digitalizer.pipeline.base: step preprocess       done in   17.9s (fei_CVI_OPACID_FEI_9788022726665)
CV

[     ok] fei/CVI_OPACID_FEI_9788022726665 — 11 MB, slk+eng, 8 min


ERROR ocrmypdf._exec.ghostscript: GPL Ghostscript 10.07.0 (2026-03-16)
Copyright (C) 2025 Artifex Software, Inc.  All rights reserved.
This software is supplied under the GNU AGPLv3 and comes with NO WARRANTY:
see the file COPYING for details.
Processing pages 1 through 58.
Page 1
Page 2
Page 3
Page 4
Page 5
Page 6
Page 7
Page 8
Page 9
Page 10
Page 11
Page 12
Page 13
Page 14
Page 15
Page 16
Page 17
Page 18
Page 19
Page 20
Page 21
Page 22
Page 23
Page 24
Page 25
Page 26
Page 27
Page 28
Page 29
Page 30
Page 31
Page 32
Page 33
Page 34
Page 35
Page 36
Page 37
Page 38
Page 39
Page 40
Page 41
Page 42
Page 43
Page 44
Page 45
Page 46
Page 47
Page 48
Page 49
Page 50
Page 51
Page 52
Page 53
Page 54
Page 55
Page 56
Page 57
Page 58

The following warnings were encountered at least once while processing this file:
	A CMap has too many code maps.

   
ERROR ocrmypdf._exec.ghostscript:  This file had errors that were repaired or ignored.
   
ERROR ocrmypdf._exec.ghostscript:  Please notify the author

[     ok] fei/CVI_OPACID_FEI_9788022726634 — 11 MB, slk+eng, 8 min


...

[     ok] mtf/CVI_OPACID_MTF_8022716286 — 20 MB, slk+eng, 15 min


ERROR ocrmypdf._exec.tesseract: [tesseract] Error during processing.
ERROR ocrmypdf._exec.ghostscript: (suppressed 4 repeated lines)
ERROR ocrmypdf._exec.ghostscript: GPL Ghostscript 10.07.0 (2026-03-16)
Copyright (C) 2025 Artifex Software, Inc.  All rights reserved.
This software is supplied under the GNU AGPLv3 and comes with NO WARRANTY:
see the file COPYING for details.
Processing pages 1 through 66.
Page 1
Page 2
Page 3
Page 4
Page 5
Page 6
Page 7
Page 8
Page 9
Page 10
Page 11
Page 12
Page 13
Page 14
Page 15
Page 16
Page 17
Page 18
Page 19
Page 20
Page 21
Page 22
Page 23
Page 24
Page 25
Page 26
Page 27
Page 28
Page 29
Page 30
Page 31
Page 32
Page 33
Page 34
Page 35
Page 36
Page 37
Page 38
Page 39
Page 40
Page 41
Page 42
Page 43
Page 44
Page 45
Page 46
Page 47
Page 48
Page 49
Page 50
Page 51
Page 52
Page 53
Page 54
Page 55
Page 56
Page 57
Page 58
Page 59
Page 60
Page 61
Page 62
Page 63
Page 64
Page 65
Page 66

The following warnings were encountered at least once while processing t

[     ok] mtf/CVI_OPACID_MTF_8022716855 — 21 MB, slk+eng, 10 min


INFO evilflowers_books_digitalizer.pipeline.base: step download         done in  193.5s (mtf_CVI_OPACID_MTF_9788080783976)
INFO evilflowers_books_digitalizer.pipeline.steps.preprocess: mtf_CVI_OPACID_MTF_9788080783976: 75 frames -> 75 pages (0 spreads split, 75 color pages kept)
INFO evilflowers_books_digitalizer.pipeline.base: step preprocess       done in   35.4s (mtf_CVI_OPACID_MTF_9788080783976)
INFO evilflowers_books_digitalizer.language: detected language sk (p=1.00)
INFO evilflowers_books_digitalizer.pipeline.base: step language         done in    8.3s (mtf_CVI_OPACID_MTF_9788080783976)
INFO evilflowers_books_digitalizer.pipeline.base: step assemble         done in   44.9s (mtf_CVI_OPACID_MTF_9788080783976)
ERROR ocrmypdf._exec.tesseract: [tesseract] Error during processing.
ERROR ocrmypdf._exec.ghostscript: (suppressed 4 repeated lines)
ERROR ocrmypdf._exec.ghostscript: GPL Ghostscript 10.07.0 (2026-03-16)
Copyright (C) 2025 Artifex Software, Inc.  All rights reserved.
This sof

[     ok] mtf/CVI_OPACID_MTF_9788080783976 — 24 MB, slk+eng, 8 min


## Report

Cumulative across all runs (the JSONL persists). Failed books: fix the cause (or just re-run — transient WebDAV errors are common) and they'll be retried since no PDF exists for them.

In [4]:
report_df = pd.read_json(report_path, lines=True)
# keep the latest attempt per book
report_df = report_df.drop_duplicates(subset=["source", "book_id"], keep="last")

ok = report_df[report_df.status == "ok"]
print(f"ok: {len(ok)} | skipped: {(report_df.status == 'skipped').sum()} "
      f"| errors: {(report_df.status == 'error').sum()}")
if len(ok):
    print(f"output: {ok.pdf_mb.sum() / 1e3:.2f} GB | "
          f"{ok.n_pages.sum():,.0f} pages | "
          f"avg {ok.minutes.mean():.1f} min/book | "
          f"languages: {ok.language.value_counts().to_dict()}")

errors = report_df[report_df.status == "error"]
errors[["source", "book_id", "error"]] if len(errors) else report_df.tail(10)

ok: 14 | skipped: 0 | errors: 0
output: 0.25 GB | 947 pages | avg 11.5 min/book | languages: {'slk+eng': 14}


,source,book_id,status,error,minutes,pdf,pdf_mb,n_frames,n_pages,language,ocr_chars
10,fei,CVI_OPACID_FEI_9788022726665,ok,NaN,7.58,/Users/jdubec/Projects/EvilFlowers/evilflowers...,10.912879,53.0,53.0,slk+eng,83013.0
11,fei,CVI_OPACID_FEI_9788022726634,ok,NaN,8.33,/Users/jdubec/Projects/EvilFlowers/evilflowers...,10.960746,58.0,58.0,slk+eng,96122.0
12,fei,CVI_OPACID_FEI_9788022726658,ok,NaN,8.74,/Users/jdubec/Projects/EvilFlowers/evilflowers...,9.686115,49.0,49.0,slk+eng,84255.0
13,fei,CVI_OPACID_FEI_9788022726689,ok,NaN,9.10,/Users/jdubec/Projects/EvilFlowers/evilflowers...,11.740024,61.0,61.0,slk+eng,102484.0
14,fei,FEI_9788089422012,ok,NaN,6.73,/Users/jdubec/Projects/EvilFlowers/evilflowers...,19.592541,77.0,77.0,slk+eng,165717.0
15,fei,CVI_OPACID_FEI_9788022726696,ok,NaN,13.11,/Users/jdubec/Projects/EvilFlowers/evilflowers...,14.146569,69.0,69.0,slk+eng,127404.0
16,fad,CVI_OPACID_FA_9788080464011,ok,NaN,14.08,/Users/jdubec/Projects/EvilFlowers/evilflowers...,22.786106,89.0,89.0,slk+eng,166269.0
17,mtf,CVI_OPACID_MTF_8022716286,ok,NaN,14.73,/Users/jdubec/Projects/EvilFlowers/evilflowers...,19.569873,64.0,65.0,slk+eng,93248.0
18,mtf,CVI_OPACID_MTF_8022716855,ok,NaN,10.22,/Users/jdubec/Projects/EvilFlowers/evilflowers...,20.730822,66.0,66.0,slk+eng,100446.0
19,mtf,CVI_OPACID_MTF_9788080783976,ok,NaN,8.25,/Users/jdubec/Projects/EvilFlowers/evilflowers...,24.031761,75.0,75.0,slk+eng,177279.0


---
**Scaling up:** set `LIMIT = None` and let it run (~40–50 h OCR + download time for all 880 books). For an unattended run, the equivalent script is:

```python
from evilflowers_books_digitalizer.batch import process_book  # + the work-list code above
```

in a plain `python` process under `caffeinate -i` / `tmux`, so the laptop doesn't sleep mid-corpus.

**Next:** classification & embeddings pipelines reading the `output/<faculty>/*.txt` sidecars.